In [1]:
!git clone https://github.com/quark28/miniml.git
%cd miniml

Cloning into 'miniml'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 141 (delta 67), reused 107 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 915.07 KiB | 12.71 MiB/s, done.
Resolving deltas: 100% (67/67), done.
/content/miniml


In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression as SklearnLogReg
from sklearn.datasets import load_breast_cancer, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss

from logistic_regression import LogisticRegression

TOL_ACC = 0.03  # допуск по accuracy между итеративными методами и sklearn


def check(name, condition, extra=""):
    status = "OK  " if condition else "FAIL"
    print(f"[{status}] {name} {extra}")
    if not condition:
        raise AssertionError(f"{name} {extra}")


X, y = load_breast_cancer(return_X_y=True)
X = StandardScaler().fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

sk_model = SklearnLogReg(max_iter=5000)
sk_model.fit(X_train, y_train)
sk_pred = sk_model.predict(X_test)
sk_acc = accuracy_score(y_test, sk_pred)

# --- 1. Базовый GD должен сходиться близко к sklearn ---
model = LogisticRegression()
model.fit(X_train, y_train, learning_type='gd', n_steps=3000, lr=0.001, lr_type='constant')
pred = model.predict(X_test)
acc = accuracy_score(y_test, pred)
check("GD: accuracy близка к sklearn", abs(acc - sk_acc) < TOL_ACC, f"(своя={acc:.4f}, sklearn={sk_acc:.4f})")

# --- 2. predict_proba: без падений, форма и диапазон корректны ---
proba = model.predict_proba(X_test)
check("predict_proba: значения в [0,1]", np.all((proba >= 0) & (proba <= 1)))
check("predict_proba: форма (n_samples,)", proba.shape == (X_test.shape[0],))

# --- 3. predict согласован с predict_proba по порогу 0.5 ---
manual_pred = np.where(proba >= 0.5, 1, 0)
check("predict() согласован с predict_proba() по порогу 0.5", np.array_equal(pred, manual_pred))

# --- 4. Все девять оптимизаторов не падают и дают разумную accuracy ---
optimizers = [
    ('sgd',      dict(learning_type='sgd', n_steps=5000, lr=0.05, lr_type='constant', batch_size=32)),
    ('sag',      dict(learning_type='sag', n_steps=3000, lr=0.05, lr_type='constant', batch_size=16)),
    ('momentum', dict(learning_type='mm', n_steps=3000, lr=0.1, lr_type='constant', gamma=0.9)),
    ('nesterov', dict(learning_type='nag', n_steps=3000, lr=0.1, lr_type='constant', gamma=0.9)),
    ('rmsprop',  dict(learning_type='rms', n_steps=3000, lr=0.1, lr_type='constant', alpha=0.9, const=1e-8)),
    ('adam',     dict(learning_type='adam', n_steps=3000, lr=0.1, lr_type='constant', gamma=0.9, alpha=0.999, const=1e-8)),
    ('nadam',    dict(learning_type='nadam', n_steps=3000, lr=0.1, lr_type='constant', gamma=0.9, alpha=0.999, const=1e-8)),
    ('adagrad',  dict(learning_type='adagrad', n_steps=3000, lr=0.5, lr_type='constant', const=1e-8)),
    ('adadelta', dict(learning_type='adadelta', n_steps=3000, lr=1.0, lr_type='constant', alpha=0.9, const=1e-5)),
]

for name, kwargs in optimizers:
    m = LogisticRegression()
    m.fit(X_train, y_train, **kwargs)
    p = m.predict(X_test)
    a = accuracy_score(y_test, p)
    check(f"{name}: accuracy разумна (>0.85)", a > 0.85, f"(acc={a:.4f})")

# --- 5. Регуляризация не ломает обучение ---
for reg, alpha in [('ridge', 0.1), ('lasso', 0.01), ('elasticnet', (0.01, 0.01))]:
    m = LogisticRegression(regularizator=reg, alpha=alpha)
    m.fit(X_train, y_train, learning_type='gd', n_steps=3000, lr=0.1, lr_type='constant')
    p = m.predict(X_test)
    a = accuracy_score(y_test, p)
    check(f"regularizator={reg}: accuracy разумна (>0.85)", a > 0.85, f"(acc={a:.4f})")

# --- 6. Конвертация классов {-1,1} <-> {0,1} ---
y_train_pm = np.where(y_train == 0, -1, 1)
y_test_pm = np.where(y_test == 0, -1, 1)

m_pm = LogisticRegression()
m_pm.fit(X_train, y_train_pm, learning_type='gd', n_steps=3000, lr=0.1, lr_type='constant')
pred_pm = m_pm.predict(X_test)

check("predict() при y в {-1,1} возвращает {-1,1}", set(np.unique(pred_pm)) <= {-1, 1})
acc_pm = accuracy_score(y_test_pm, pred_pm)
check("{-1,1}-режим даёт ту же accuracy, что {0,1}-режим", abs(acc_pm - acc) < TOL_ACC,
      f"(pm={acc_pm:.4f}, 01={acc:.4f})")

# --- 7. predict_proba на исходном X (без bias) не падает по размерности (проверка исправленного бага) ---
try:
    proba_check = model.predict_proba(X_test)
    check("predict_proba() не падает на исходном X (bias добавляется внутри)", True)
except ValueError as e:
    check("predict_proba() не падает на исходном X", False, f"({e})")

# --- 8. Сравнение log_loss со sklearn (насколько калибрована модель) ---
sk_proba = sk_model.predict_proba(X_test)[:, 1]
my_logloss = log_loss(y_test, proba)
sk_logloss = log_loss(y_test, sk_proba)
check("log_loss своей модели разумно близок к sklearn", abs(my_logloss - sk_logloss) < 0.2,
      f"(своя={my_logloss:.4f}, sklearn={sk_logloss:.4f})")

# --- 9. Пороговое значение treshold работает ---
pred_strict = model.predict(X_test, treshold=0.9)
n_positive_strict = (pred_strict == 1).sum()
n_positive_default = (pred == 1).sum()
check("Более строгий treshold даёт меньше/равно positive-предсказаний",
      n_positive_strict <= n_positive_default,
      f"(treshold=0.9: {n_positive_strict}, treshold=0.5: {n_positive_default})")

# --- 10. Итоговая сводка по всем прогонам ---
print(f"sklearn accuracy (baseline):        {sk_acc:.4f}")
print(f"своя GD accuracy:                    {acc:.4f}")
print(f"своя log_loss vs sklearn:            {my_logloss:.4f} vs {sk_logloss:.4f}")

[OK  ] GD: accuracy близка к sklearn (своя=0.9825, sklearn=0.9825)
[OK  ] predict_proba: значения в [0,1] 
[OK  ] predict_proba: форма (n_samples,) 
[OK  ] predict() согласован с predict_proba() по порогу 0.5 
[OK  ] sgd: accuracy разумна (>0.85) (acc=0.9766)
[OK  ] sag: accuracy разумна (>0.85) (acc=0.9942)


/content/miniml/logistic_regression.py:78: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X @ w))


[OK  ] momentum: accuracy разумна (>0.85) (acc=0.9474)


/content/miniml/logistic_regression.py:78: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X @ w))


[OK  ] nesterov: accuracy разумна (>0.85) (acc=0.9591)
[OK  ] rmsprop: accuracy разумна (>0.85) (acc=0.9474)
[OK  ] adam: accuracy разумна (>0.85) (acc=0.9298)
[OK  ] nadam: accuracy разумна (>0.85) (acc=0.9532)
[OK  ] adagrad: accuracy разумна (>0.85) (acc=0.9766)
[OK  ] adadelta: accuracy разумна (>0.85) (acc=0.9708)


...

[OK  ] regularizator=ridge: accuracy разумна (>0.85) (acc=0.9825)


/content/miniml/logistic_regression.py:78: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X @ w))


[OK  ] regularizator=lasso: accuracy разумна (>0.85) (acc=0.9474)
[OK  ] regularizator=elasticnet: accuracy разумна (>0.85) (acc=0.9649)


/content/miniml/logistic_regression.py:78: RuntimeWarning: overflow encountered in exp
  return 1 / (1 + np.exp(-X @ w))


[OK  ] predict() при y в {-1,1} возвращает {-1,1} 
[OK  ] {-1,1}-режим даёт ту же accuracy, что {0,1}-режим (pm=0.9532, 01=0.9825)
[OK  ] predict_proba() не падает на исходном X (bias добавляется внутри) 
[OK  ] log_loss своей модели разумно близок к sklearn (своя=0.0741, sklearn=0.0598)
[OK  ] Более строгий treshold даёт меньше/равно positive-предсказаний (treshold=0.9: 102, treshold=0.5: 107)
sklearn accuracy (baseline):        0.9825
своя GD accuracy:                    0.9825
своя log_loss vs sklearn:            0.0741 vs 0.0598
